# Semaine 8 — Jour 5 : Gouvernance des versions et décisions

Relier dataset, configuration, rapports et approbation dans une chaîne de preuves.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour le parcours complet avec entretien et challenge. J1 conserve sa durée et son contenu d’origine; il peut être réparti sur plusieurs sessions.

Prérequis : J1, Python 3.11+, JSON et tests backend. Aucun accès fournisseur, aucune clé LLM et aucune infrastructure externe nécessaires.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks : `notebooks/week08/S8_J5_gouvernance.ipynb` et `notebooks/week08/teacher/S8_J5_gouvernance_teacher.ipynb` depuis la racine du dépôt.

Critère de réussite : Décrire responsabilités, rollback et limites des empreintes.

Les observations et prix sont synthétiques; les résultats mesurent le fonctionnement du protocole local, pas un fournisseur réel.


# Objectifs mesurables

- Relier dataset, configuration, rapports et approbation dans une chaîne de preuves.
- Refuser une approbation périmée ou portant sur une autre configuration.
- Décrire responsabilités, rollback et limites des empreintes.

## Preuves attendues

Une sortie reproductible, une interprétation de son dénominateur et une limite opérationnelle correctement expliquée.


# S8 J5 — Gouvernance des versions et décisions


## 1. Gouverner une modification concrète
Un système IA change lorsque le modèle, le prompt, les outils, les données ou les droits changent. Conserver seulement le commit du backend laisse une partie du comportement sans version. La gouvernance doit permettre de répondre à une question simple : qui a autorisé quelle configuration, sur quelles preuves et avec quelle stratégie de retour ?

Le manifeste du cours décrit release_id, propriétaire, relecteur, empreinte de configuration, date limite, revue de risque et référence de rollback. Les identités sont fictives et les approbations sont des données de test. Le programme n’authentifie aucune personne et ne déploie rien.

## 2. Relier les objets sans confondre intégrité et confiance
Le dataset de J1 possède une empreinte canonique. Le lot de performance reprend cette empreinte et ajoute celles du workload et de la configuration. Le rapport final inclut une empreinte de toutes les preuves reçues. Un changement devient détectable quand on compare à une valeur de référence connue.

Une empreinte ne prouve ni l’auteur ni la véracité des mesures. Un acteur capable de remplacer fichier et hash peut fabriquer une cohérence locale. En exploitation, associer collecte contrôlée, artefacts protégés, provenance du pipeline et signatures selon le besoin. La liste des relecteurs du lab vient d’une politique locale; elle n’est pas un annuaire de confiance.

## 3. L’approbation porte sur un contenu précis
Le contrôleur compare config_hash du manifeste à la configuration observée dans les runs. Une différence produit configuration_changed et bloque la gouvernance. Il vérifie aussi approved, le relecteur autorisé, la séparation propriétaire/relecteur et la validité à la date explicite de décision.

La date valid_until est inclusive : une approbation au 20 novembre reste valable pour une décision datée du 20 novembre. Le replay utilise le 20 octobre 2026 par défaut pour reproduire le cours. Pour une décision opérationnelle, fournir la date réelle et une source d’horloge fiable. Changer une date d’entrée pour contourner une expiration ne serait pas un contrôle fiable.

## 4. Séparer les responsabilités
Le propriétaire prépare et documente la version. Le relecteur vérifie les preuves et autorise le périmètre prévu. L’exploitation surveille les signaux et peut déclencher le rollback selon le runbook. La séparation de fonctions du lab est une règle pédagogique; le niveau d’exigence réel dépend de l’organisation et du risque.

Éviter deux extrêmes : tout laisser au développeur sans preuve, ou multiplier les documents sans pouvoir relier une décision à la configuration livrée. Une exception doit avoir un responsable, un périmètre, une date et un traitement du risque résiduel. Le moteur du cours n’autorise aucune exception automatique aux cas critiques.

## 5. Préparer le retour avant le déploiement
rollback_ref identifie une version antérieure fictive. Sa présence rend le manifeste complet mais ne prouve pas la possibilité d’un retour effectif. Vérifier compatibilité des contrats, état durable, prompts, index et outils. Revenir au code précédent ne restaure pas une donnée supprimée ou un message déjà publié.

Le cours livre un modèle de registre et un runbook, sans opérateur de déploiement. Le projet final doit montrer un refus cohérent quand les preuves sont insuffisantes. Une décision locale accepted n’est pas une autorisation réelle d’exécuter une action dans un compte externe.

## 6. Rétention et confidentialité
Définir les informations à garder pour expliquer une décision et celles à exclure. Conserver rapports, références et hashes peut suffire pour des analyses agrégées; les exemples exacts demandent parfois un stockage séparé à accès limité. Une empreinte de texte sensible ne rend pas forcément ce texte anonyme, notamment si son domaine de valeurs est petit.

Le registre synthétique ne contient pas de données personnelles réelles. Les durées de rétention et responsabilités du modèle documentaire sont des choix pédagogiques à adapter, sans présenter ce cours comme une preuve de conformité juridique.

## À retenir
Une décision traçable relie des artefacts versionnés, des rôles et un périmètre. Le hash explique ce qui a changé; une chaîne de confiance explique pourquoi on peut croire les preuves.


# Lab exécutable

Cellules autonomes à partir du dépôt; aucune requête réseau.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week08/ops.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from copy import deepcopy
from examples.week08.demo import load
from examples.week08.ops import govern
data = load("baseline")
config_hash = data["runs"][0]["config_hash"]
valid = govern(data["manifest"], config_hash, data["policy"], "2026-10-20")
assert valid["accepted"]
changed = deepcopy(data["manifest"])
changed["config_hash"] = "another-configuration"
invalid = govern(changed, config_hash, data["policy"], "2026-10-20")
assert not invalid["accepted"] and "configuration_changed" in invalid["reasons"]
print(valid, invalid)

# Exercices

Conserver une première réponse avant de lire la correction.

## Exercice 1 — Identifier la configuration

Lister cinq éléments de comportement à versionner au-delà du commit Python.

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 2 — Provoquer une décision obsolète

Changer config_hash du manifeste sans modifier les runs. Expliquer le résultat.

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 3 — Analyser le rollback

Pourquoi rollback_ref non vide ne prouve-t-il pas un retour opérationnel ?

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.


# Entretien

Répondre en deux minutes par question, avec un exemple et une limite.

## Question 1

Que prouve un hash ?

## Question 2

Quels changements nécessitent une nouvelle revue ?


# Challenge

Tester les quatre refus suivants : relecteur inconnu, propriétaire égal au relecteur, approbation absente et date expirée. Conserver les raisons distinctes.

## Critères

Reproductibilité, garde vérifiée, distinction entre résultat mesuré et garantie non démontrée.


# Références — J5

Sources de référence consultées le 20 octobre 2026. Le lab implémente ses propres contrats pédagogiques; il n’affirme pas déployer les outils cités.

- [OpenTelemetry : concepts d’observabilité](https://opentelemetry.io/docs/concepts/observability-primer/) — signaux et diagnostic.
- [OpenTelemetry : traces](https://opentelemetry.io/docs/concepts/signals/traces/) — contexte et spans pour une intégration réelle.
- [OWASP : injection de prompt, édition 2025](https://genai.owasp.org/llmrisk/llm01-prompt-injection/) — référence datée, sans prétendre être la dernière édition.
- [OWASP : excessive agency, édition 2025](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/) — permissions et autonomie.
- [Python : Decimal](https://docs.python.org/3/library/decimal.html) — calcul décimal pour les coûts.
- [Python : unittest](https://docs.python.org/3/library/unittest.html) — reproduction des contrôles.

Lire prioritairement les sources liées au thème du jour. Les seuils, tarifs et identités du lab sont fictifs et ne proviennent pas de ces documents. Le calcul nearest-rank est défini explicitement dans le chapitre et le code.
